<div align="right"><a href="S12-Filter-Structures.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). Beachte bitte, dass zur Ausführung von MATLAB in Notebooks zunächst die [MATLAB API für Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) installiert werden muss. Die Installation wird [hier](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html) beschrieben. <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/L12-Filterstrukturen.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L11 Korrelationsfolgen](./L11-Korrelationsfolgen.ipynb) | [Aufgabe 12](./A12-Filterstrukturen.ipynb) | [Inhalt / Übersicht](./00-MATLAB-Notebooks-Overview-DE.ipynb) | Weiter: [L13 Entwurf rekursiver Filter](./L13-Entwurf-rekursiver-Filter.ipynb)

</div>

---


# Aufgabe: Filterstrukturen

Bis jetzt wurden LTI-Systeme durch die Koeffizientenvektoren `b` und `a` beschrieben, was der üblichen Darstellung mittels System- bzw. Z-Übertragungsfunktion entspricht:

$$
H(z) = \frac{b_0 + b_1 z^{-1} + b_2 z^{-2} + \cdots}{1+ a_1 z^{-1} + a_2 z^{-2} + \cdots} \,\,\,.
$$

Eine andere (äquivalente) Darstellung ist über Pole und Nullstellen möglich:

$$
H(z) = b_0 \frac{(z-z_{0,1})(z-z_{0,2})(z-z_{0,3})\cdots}{(z-z_{\infty,1})(z-z_{\infty,2})(z-z_{\infty,3})\cdots} \qquad.
$$

LTI-Systeme können mit *kanonischen* Strukturen realisiert werden. Üblicherweise bezeichnet eine kanonische Struktur eine Realisierungsform mit minimaler Anzahl an Speicherelementen. Siehe Kapitel 4.1 für die Beschreibung der einzelnen Implementierungsarten. Die in dieser Aufgabe betrachtete *dritte kanonische* Struktur erlaubt es, die Systemfunktion direkt in eine Pol-Nullstellen-Darstellung zu transformieren.

Die einzelnen Pole bzw. Nullstellen werden dabei in Abschnitte maximal zweiter Ordnung gruppiert (SOS = Second Order Section) und in einer Kaskade hintereinandergeschaltet. Die Hauptvorteile dieser Struktur sind die Flexibilität in der Reihenfolge der Abschnitte sowie das gute Verhalten bei der Verwendung quantisierter Parameter.

(Hinweis: Zum Thema Parameterquantisierung siehe Abschnitt 4.4 bzw. Aufgabe dsv4a04e.)

Die MATLAB-Funktion `tf2zp()` konvertiert die Polynomform in die Pol-Nullstellen-Darstellung. Mit Hilfe der Pol-Nullstellen-Darstellung kann die Funktion `zp2sos()` die Second Order Sections der dritten kanonischen Form berechnen.

---

a)

Schreiben Sie eine neue MATLAB-Funktion `[B, A] = lcascade(b, a)`, die aus den Filtervektoren `b` und `a` Abschnitte zweiter Ordnung erstellt. Die Koeffizienten der Second Order Sections sollen mit `tf2zp()` und `zp2sos()` berechnet werden und als Zeilenvektoren in den Matrizen `B` und `A` gespeichert werden.

In [ ]:
# BEGIN SOLUTION
function [B, A] = lcascade(b, a)
    [z, p, k] = tf2zp(b, a);
    sos = zp2sos(z, p, k);
    B = sos(:, 1:3);  % numerator coefficients
    A = sos(:, 4:6);  % denominator coefficients
# END SOLUTION

b)

Berechnen Sie (mithilfe von `lcascade()`) die Filterkoeffizienten der Second-Order-Sections eines Cauer-Tiefpasses 7. Ordnung mit folgenden Spezifikationen:

- $3\,\text{dB}$-Grenzfrequenz: $0{,}4\,\pi$
- Dämpfung im Durchlassbereich: $R_P = 0{,}1\,\text{dB}$
- Dämpfung im Sperrbereich: $R_S = 40\,\text{dB}$

Die Toleranzparameter $R_S$ bzw. $R_P$ sind wie folgt definiert:

$$
    R_P = -20 \log_{10}(1 - \delta_D), \quad
    R_S = -20 \log_{10}(\delta_S),
$$
wobei $\delta_D$ bzw. $\delta_S$ die Toleranzparameter gemäß Kapitel 4 darstellen.

Die MATLAB-Funktion `ellip(7, 0.1, 40, 0.4)` liefert die Filterkoeffizienten:
$$
    [b, a] = ellip(7, 0.1, 40, 0.4);
    [B, A] = lcascade(b, a);
$$
Berechnen Sie die Übertragungsfunktion des Gesamtsystems gemäß:
$$
    H(z) = \prod_{i=1}^{p} H_i(z).
$$

In [ ]:
# BEGIN SOLUTION
[b, a] = ellip(7, 0.1, 40, 0.4);  % Filter entwerfen
[B, A] = lcascade(b, a);          % Umwandlung in 3. kanonische Form
[Hg, W] = freqz(b, a, 512);       % Übertragungsfunktion des Gesamtsystems
[m, n] = size(B);                 
Hi = zeros(512, m);

for k = 1:m                    
    Hi(:, k) = freqz(B(k, :), A(k, :), 512);  % Frequenzgang jeder Sektion
end

% Produkt der Übertragungsfunktionen der Teilsysteme
H3k = Hi(:, 1) .* Hi(:, 2) .* Hi(:, 3) .* Hi(:, 4);

% # Grafische Ausgabe # %
figure;
plot(W/pi, 20*log10(abs(H3k)), '--b');       % Teilsysteme (gestrichelt)
hold on;
plot(W/pi, 20*log10(abs(Hg)), '-.b');        % Gesamtsystem (strichpunkt)
hold off;
grid; 
xlabel('\Omega/\pi (Nyquist = 1)'); 
ylabel('|H(e^{j\Omega})| in dB');
title('-- Gesamtsystem, -.Produkt der Teilsysteme');
# END SOLUTION